In [1]:


import sys
sys.path.insert(0, '..')

import json
import numpy as np
from pathlib import Path
from collections import Counter

DATA_DIR = Path('../data/raw')

# ─── Document counts by source ───
print('═' * 60)
print('  📊 DATA EXPLORATION')
print('═' * 60)

stats = {}
for source_dir in sorted(DATA_DIR.iterdir()):
    if not source_dir.is_dir():
        continue
    files = list(source_dir.glob('*.json')) + list(source_dir.glob('*.md'))
    char_counts = []
    for f in files:
        try:
            if f.suffix == '.json':
                data = json.loads(f.read_text(encoding='utf-8'))
                content = data.get('content', '')
            else:
                content = f.read_text(encoding='utf-8')
            char_counts.append(len(content))
        except Exception:
            continue
    stats[source_dir.name] = {'files': len(files), 'chars': char_counts}

print(f'\n{"Source":<10} {"Files":<8} {"Total Chars":<15} {"Avg Length":<12} {"Est Chunks":<10}')
print('─' * 55)
for name, s in stats.items():
    total = sum(s['chars'])
    avg = int(np.mean(s['chars'])) if s['chars'] else 0
    print(f'{name:<10} {s["files"]:<8} {total:>12,}   {avg:>9,}   {total // 512:>7}')

total_chars = sum(sum(s['chars']) for s in stats.values())
total_files = sum(s['files'] for s in stats.values())
print('─' * 55)
print(f'{"TOTAL":<10} {total_files:<8} {total_chars:>12,}   {"":>9}   {total_chars // 512:>7}')

# ─── Length distribution ───
all_lengths = []
for s in stats.values():
    all_lengths.extend(s['chars'])
arr = np.array(all_lengths)

print(f'\n\nDocument Length Statistics:')
print(f'  Min:    {arr.min():>8,} chars')
print(f'  25th:   {int(np.percentile(arr, 25)):>8,} chars')
print(f'  Median: {int(np.median(arr)):>8,} chars')
print(f'  75th:   {int(np.percentile(arr, 75)):>8,} chars')
print(f'  Max:    {arr.max():>8,} chars')
print(f'  Mean:   {arr.mean():>8,.0f} chars')

# ─── Sample document ───
wiki_dir = DATA_DIR / 'wiki'
if wiki_dir.exists():
    sample_file = sorted(wiki_dir.glob('*.json'))[0]
    sample = json.loads(sample_file.read_text(encoding='utf-8'))
    print(f'\n\n{"═" * 60}')
    print(f'Sample Document: {sample.get("title", sample_file.stem)}')
    print(f'Length: {len(sample.get("content", "")):,} chars')
    print(f'{"─" * 60}')
    print(sample.get('content', '')[:400] + '...')

# ─── Topics ───
if wiki_dir.exists():
    titles = []
    for f in sorted(wiki_dir.glob('*.json')):
        d = json.loads(f.read_text(encoding='utf-8'))
        titles.append(d.get('title', f.stem))
    print(f'\n\nWikipedia Topics ({len(titles)}):')
    for i, t in enumerate(titles, 1):
        print(f'  {i:>2}. {t}')

# ─── Eval set ───
eval_path = Path('../data/eval_sets/eval_set.json')
if eval_path.exists():
    with open(eval_path) as f:
        eval_set = json.load(f)
    types = Counter(item.get('type', '?') for item in eval_set)
    print(f'\n\nEval Set: {len(eval_set)} questions')
    print(f'Types: {dict(types)}')

print(f'\n\n✅ Data exploration complete')

════════════════════════════════════════════════════════════
  📊 DATA EXPLORATION
════════════════════════════════════════════════════════════

Source     Files    Total Chars     Avg Length   Est Chunks
───────────────────────────────────────────────────────
docs       10            101,475      10,147       198
papers     56             73,203       1,307       142
wiki       30          1,048,273      34,942      2047
───────────────────────────────────────────────────────
TOTAL      96          1,222,951                  2388


Document Length Statistics:
  Min:         491 chars
  25th:      1,288 chars
  Median:    1,654 chars
  75th:     14,836 chars
  Max:     114,086 chars
  Mean:     12,739 chars


════════════════════════════════════════════════════════════
Sample Document: Attention (machine learning)
Length: 25,450 chars
────────────────────────────────────────────────────────────
In machine learning, attention is a method that determines the importance of each component i